In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [3]:
### Load the trained model , scaler pickle, onehot
model = load_model('model.h5')

## load the encoder and scaler

with open('ohe_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)



In [4]:
## Example input data 
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
    
}

In [5]:
## One Hpt Enocde 'GeoGraphy'ArithmeticError
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=label_encoder_geo.get_feature_names_out(['Geography']))


f:\PYTHON_AI\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [6]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [7]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [8]:
## Encode my cateogrical variable

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [9]:
## Combine ohe-hot encoded data

input_df = pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [10]:
## Scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53537913,  0.94880848,  0.0893036 , -0.63493589, -0.22028801,
         0.65234719,  0.64298333,  0.9900495 , -0.95357329,  1.00501256,
        -0.57542576, -0.58312392]])

In [11]:
## Predict churn
predict = model.predict(input_scaled)
predict

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


array([[0.23285715]], dtype=float32)

In [12]:
predict_prob = predict[0][0]

In [13]:
predict_prob

np.float32(0.23285715)

In [14]:
if predict_prob> 0.5 :
    print('The Customer Is Likely to Churn')
else:
    print('The Customer Is Not Likely to Churn')

The Customer Is Not Likely to Churn
